In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go

from utils import *

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.set_option("display.max_columns", None)

# Load data and take an initial look

As a first step, we load the new data set with all wind turbines from the market master data register. This is already available in a special format for tables: Parquet. 

By executing the cell below, you load the data into Pandas, as you did in the last exercise. In addition, we also execute `.info()` to get information about our columns.

In [ ]:
MaStRWind = pd.read_parquet("MaStRWind.parquet")
MaStRWind.info()

That really is a hell of a lot of columns... I can't even imagine what's behind many of the names!

Fortunately, we already had this top function in Pandas: `.head(n)`.

Here is the explanation again:
This allows you to display the first n rows of your table, i.e. the head. Logical, as “head” already says. So if we want to see the first 10 rows, we write `.head(n=10)`. If we want the first 50 lines, then `.head(n=50)`. The same works if you want to see the last lines from the bottom. You can use `.tail(n)` for this. So if you want to see the bottom 10 rows, what do you use? Try it out!

In [ ]:
MaStRWind.head(n=5)

Here is a small function that we have prepared to let a little annoying data processing run in the background. Please ignore all the error messages. Unfortunately Claudius couldn't get rid of them, he still has a lot to learn...

In [ ]:
MaStRWind = data_preparation(MaStRWind)

# Visualize and analyze data

I don't know about you, but I always find it difficult to work with such huge tables. It's always easier for me to recognize things when I have pictures in front of me. If you feel the same way, we have something for you to try out in the next few cells. 

### Weekly activities

As a bit of fun, let's see on which days of the week the most wind turbines are actually put into operation in each federal state. 

In [ ]:
MaStRWind_grouped = MaStRWind.loc[MaStRWind["Inbetriebnahmejahr"]>1990].groupby(by=["Bundesland","Inbetriebnahmewochentag"]).sum()["Anlagenzahl"].reset_index()
#MaStRWind_grouped["Anlagenzahl"] = MaStRWind_grouped["Anlagenzahl"] / 1000000
MaStRWind_grouped_pivot = MaStRWind_grouped.pivot(columns='Inbetriebnahmewochentag',index='Bundesland',values='Anlagenzahl')

weekdays = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

plt.figure(figsize=(8,8))
ax = sns.heatmap(MaStRWind_grouped_pivot, annot=True, fmt=".2f", cbar_kws={'label': 'commissioning of wind power plants per weekday [-]'})
ax.set_xticklabels(weekdays)
plt.xticks(rotation = 45)
plt.show()

Thursday seems to be the busiest day everywhere. Maybe people just want to tidy up their building site on Friday so they can have a good start to the weekend? I don't know... And why is that, do we see that in the data? Maybe you can share your thoughts on this with us? And what's actually going on with the offshore wind turbines?

### Expansion of the federal states since 1990 

Next, let's take a look at the expansion since 1990. To do this, we will again look at all the federal states and the offshore sector individually. However, we are not looking at the pure number of turbines, but at the installed capacity. In other words, the combined output of all wind turbines commissioned in a given year and federal state. That will certainly be quite a picture...

In [ ]:
MaStRWind_grouped = MaStRWind.loc[MaStRWind["Inbetriebnahmejahr"]>1990].groupby(by=["Bundesland","Inbetriebnahmejahr"]).sum()["Nettonennleistung"].reset_index()
MaStRWind_grouped["Nettonennleistung"] = MaStRWind_grouped["Nettonennleistung"] / 1000
MaStRWind_grouped_pivot = MaStRWind_grouped.pivot(columns='Inbetriebnahmejahr',index='Bundesland',values='Nettonennleistung')

plt.figure(figsize=(30,8))
sns.heatmap(MaStRWind_grouped_pivot, annot=True, fmt=".2f", cbar_kws={'label': 'commissioning of power generation capacity [MW/a]'}, cmap="viridis")

What happened from 2020 onwards? Things suddenly stopped progressing at all... Can you perhaps help me with the reason for this?

### Spatial correlations

Now let's take a look at how the wind turbines are distributed in Germany. Specifically, where large turbines are located and where small ones are. To do this, we will plot all the available analogs over their coordinates, let the size of the points be determined by the rotor diameters and the color according to the federal states. I can't wait to see what that looks like!

In [ ]:
plt.figure(figsize=(12,12))
sns.scatterplot(
    data=MaStRWind.loc[MaStRWind["Inbetriebnahmejahr"]>1990], 
    x="Laengengrad", y="Breitengrad", 
    hue="Bundesland", size="Rotordurchmesser", #style="Bundesland", 
    palette="husl"
)

Oops, there are some plants in there that aren't even in Germany... Where does that come from? Do you have any ideas?

Next, we want to take a closer look at the technical parameters that Marie just presented to us on a similar map. Specifically hub height and rotor diameter. Can you recognize a trend in the illustration that you remember from the presentation? Two little tip words: *North-South*

In [ ]:
plt.figure(figsize=(12,12))
sns.scatterplot(
    data=MaStRWind.loc[MaStRWind["Inbetriebnahmejahr"]>1990], 
    x="Laengengrad", y="Breitengrad", 
    hue="Nabenhoehe", 
    size="Rotordurchmesser", 
    #style="Bundesland", 
    #palette="husl", 
    sizes=(10, 150),
)

At the end, let's take another look at the development over time. So let's take another look at rotor diameter and hub height, but now not how they are distributed in Germany, but how they have changed over time. Can you describe what you see there?

In [ ]:
plt.figure(figsize=(16,12))
sns.scatterplot(
    data=MaStRWind.loc[(MaStRWind["Inbetriebnahmejahr"]>1990) & (MaStRWind["Bundesland"]!="Seelage")], 
    x="Inbetriebnahmedatum", y="Nabenhoehe", 
    hue="Bundesland", size="Rotordurchmesser", sizes=(20, 200)
)

And a little gimmick at the very end. Here we have prepared something interactive for you. After executing the cell, you can then view all the wind turbines from the MaStR on a map like Google Maps. Also with zooming in and out and additional information when hovering with the mouse! 

Unfortunately, the satellite service is a bit buggy today, so we can't switch to the other background... Sorry :/

In [ ]:
MaStRWind_wms = MaStRWind[MaStRWind["Rotordurchmesser"].notna()]
MaStRWind_wms = MaStRWind_wms.loc[MaStRWind_wms["Inbetriebnahmejahr"]>1990]
MaStRWind_wms.sort_values(by=["Inbetriebnahmedatum"], inplace=True)

fig = go.Figure(px.scatter_mapbox(MaStRWind_wms, 
            lat="Breitengrad", 
            lon="Laengengrad", 
            color="Nettonennleistung",
            #size="Rotordurchmesser",
            #size_max=40,
            hover_name="EinheitMastrNummer", 
            hover_data=["Inbetriebnahmedatum","Hersteller","Typenbezeichnung","Nettonennleistung","Nabenhoehe","Rotordurchmesser"],
            color_continuous_scale="jet", 
            zoom=4,
            height=900,
            mapbox_style="open-street-map",
            #animation_frame="Inbetriebnahmejahr",
            )
        )      

fig.update_layout(
    margin = {'l':0, 'r':0, 'b':0, 't':0},
    coloraxis_colorbar=dict(
        title="Nettonennleistung [kW]",
    ),
    title="Windenergieanlagenbestand nach Leistung und Rotordurchmesser",
)

fig.show()